# Dataset analysis for Caltech101 and Enrico

This notebook keeps the code compact and focuses on a few paper-friendly views: Caltech101 summary statistics, then Enrico screenshots vs wireframes with UMAP-based similarity and overlap metrics.

In [ ]:
!pip install -q umap-learn

import sys
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from torch.utils.data import ConcatDataset, DataLoader, Subset
from torchvision import datasets
from torchvision.transforms import v2
from umap import UMAP

for candidate in [Path.cwd(), Path.cwd() / 'src', Path.cwd() / 'src' / 'data', Path.cwd() / 'src' / 'utils']:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from natural_images import AugmentationWrapper, make_caltech_base_transform, stratified_three_way_split
from screen_images import CustomEnricoDataset, plot_image_list
from training import mean_and_std_for_normalization

SEED = 317
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
sns.set_theme(style='whitegrid')

In [ ]:
def extract_labels(dataset):
    if isinstance(dataset, Subset):
        parent_labels = extract_labels(dataset.dataset)
        return np.asarray(parent_labels)[dataset.indices]
    if isinstance(dataset, ConcatDataset):
        return np.concatenate([extract_labels(child) for child in dataset.datasets])
    if hasattr(dataset, 'targets'):
        labels = dataset.targets
        return labels.cpu().numpy() if torch.is_tensor(labels) else np.asarray(labels)
    if hasattr(dataset, 'labels'):
        labels = dataset.labels
        return labels.cpu().numpy() if torch.is_tensor(labels) else np.asarray(labels)
    if hasattr(dataset, 'y'):
        labels = dataset.y
        return labels.cpu().numpy() if torch.is_tensor(labels) else np.asarray(labels)
    return np.asarray([dataset[index][1] for index in range(len(dataset))])

def class_count_frame(dataset, class_names):
    labels = extract_labels(dataset)
    counts = pd.Series(labels).value_counts().sort_index()
    name_lookup = {index: name for index, name in enumerate(class_names)}
    return pd.DataFrame({
        'class_index': counts.index,
        'class_name': [name_lookup.get(index, str(index)) for index in counts.index],
        'count': counts.values,
    })

def split_summary_frame(train_ds, val_ds, test_ds):
    splits = [('train', train_ds), ('val', val_ds), ('test', test_ds)]
    total = sum(len(ds) for _, ds in splits)
    return pd.DataFrame({
        'split': [name for name, _ in splits],
        'samples': [len(ds) for _, ds in splits],
        'share_%': [round(len(ds) / total * 100, 2) for _, ds in splits],
    })

def plot_class_count_histogram(counts, title):
    plt.figure(figsize=(8, 4))
    sns.histplot(counts.values, bins=min(20, max(5, len(counts) // 4)), color='#2a9d8f')
    plt.title(title)
    plt.xlabel('Images per class')
    plt.ylabel('Number of classes')
    plt.tight_layout()
    plt.show()

def show_sample_grid(dataset, class_names=None, title='Samples', n_samples=9):
    sample_count = min(n_samples, len(dataset))
    rng = np.random.default_rng(SEED)
    indices = rng.choice(len(dataset), size=sample_count, replace=False)
    images = []
    titles = []
    for index in indices:
        image, label = dataset[index]
        image_array = image.permute(1, 2, 0).cpu().numpy() if torch.is_tensor(image) else np.asarray(image)
        images.append(image_array)
        if class_names is not None and label < len(class_names):
            titles.append(class_names[label])
        else:
            titles.append(str(label))
    print(title)
    plot_image_list(images, titles=titles, cols=3)

def dataset_to_matrix(dataset, resize=(48, 48), batch_size=128):
    analysis_transform = v2.Compose([
        v2.Resize(resize),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
    ])
    loader = DataLoader(AugmentationWrapper(dataset, analysis_transform), batch_size=batch_size, shuffle=False, num_workers=0)
    feature_blocks = []
    label_blocks = []
    for images, labels in loader:
        feature_blocks.append(images.flatten(1).cpu().numpy())
        label_blocks.append(labels.cpu().numpy())
    return np.concatenate(feature_blocks), np.concatenate(label_blocks)

def project_with_umap(features, seed=SEED):
    scaled = StandardScaler().fit_transform(features)
    n_components = min(50, scaled.shape[0] - 1, scaled.shape[1])
    reduced = PCA(n_components=n_components, random_state=seed).fit_transform(scaled) if n_components >= 2 else scaled
    projector = UMAP(n_neighbors=15, min_dist=0.15, metric='euclidean', random_state=seed)
    return projector.fit_transform(reduced)

def plot_umap(coords, labels, class_names, title):
    frame = pd.DataFrame(coords, columns=['umap_1', 'umap_2'])
    frame['label'] = labels
    frame['class_name'] = frame['label'].map(lambda index: class_names[index])
    palette = dict(zip(class_names, sns.color_palette('tab20', n_colors=len(class_names))))

    plt.figure(figsize=(11, 8))
    sns.scatterplot(
        data=frame,
        x='umap_1',
        y='umap_2',
        hue='class_name',
        hue_order=class_names,
        palette=palette,
        s=18,
        alpha=0.8,
        linewidth=0,
    )
    plt.title(title)
    plt.xlabel('UMAP-1')
    plt.ylabel('UMAP-2')
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='Class', fontsize=7)
    plt.tight_layout()
    plt.show()

def summarize_umap(coords, labels, class_names, k=15):
    labels = np.asarray(labels)
    n_classes = len(class_names)
    centroids = np.vstack([coords[labels == class_index].mean(axis=0) for class_index in range(n_classes)])
    centroid_distance = pd.DataFrame(pairwise_distances(centroids), index=class_names, columns=class_names)

    n_neighbors = min(k + 1, len(coords))
    neighbors = NearestNeighbors(n_neighbors=n_neighbors).fit(coords).kneighbors(return_distance=False)[:, 1:]
    overlap = np.zeros((n_classes, n_classes), dtype=float)
    counts = np.bincount(labels, minlength=n_classes)

    for row_index, source_class in enumerate(labels):
        neighbor_labels = labels[neighbors[row_index]]
        overlap[source_class] += np.bincount(neighbor_labels, minlength=n_classes) / max(1, len(neighbor_labels))

    overlap = overlap / counts[:, None]
    overlap_df = pd.DataFrame(overlap, index=class_names, columns=class_names)
    purity = pd.Series(np.diag(overlap), index=class_names, name=f'purity@{k}').sort_values(ascending=False)

    pair_rows = []
    for i in range(n_classes):
        for j in range(i + 1, n_classes):
            pair_rows.append({
                'class_a': class_names[i],
                'class_b': class_names[j],
                'neighbor_overlap': 0.5 * (overlap[i, j] + overlap[j, i]),
                'centroid_distance': centroid_distance.iat[i, j],
            })

    pairwise = pd.DataFrame(pair_rows)
    return {
        'centroid_distance': centroid_distance,
        'overlap': overlap_df,
        'purity': purity,
        'pairwise': pairwise,
    }

def print_umap_summary(summary, title):
    print(f'\n{title}')
    print('Per-class neighborhood purity')
    display(summary['purity'].to_frame())
    print('Class-by-class overlap matrix')
    display(summary['overlap'].round(3))
    print('Class centroid distances in UMAP space')
    display(summary['centroid_distance'].round(3))
    print('Most overlapping class pairs')
    display(summary['pairwise'].sort_values('neighbor_overlap', ascending=False).head(10).round(3))
    print('Closest class centroids')
    display(summary['pairwise'].sort_values('centroid_distance').head(10).round(3))

## Caltech101 summary

The goal here is to keep the stats compact but useful: split balance, class balance, a few sample images, and the original-train mean/std used for standardization.

In [ ]:
caltech_dataset = datasets.Caltech101(root='./data', download=True)
caltech_train_raw, caltech_val_raw, caltech_test_raw = stratified_three_way_split(
    dataset=caltech_dataset,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    random_state=SEED,
)

caltech_labels = extract_labels(caltech_dataset)
caltech_class_names = [str(index) for index in sorted(np.unique(caltech_labels))]
caltech_train_counts = class_count_frame(caltech_train_raw, caltech_class_names)
caltech_split_counts = split_summary_frame(caltech_train_raw, caltech_val_raw, caltech_test_raw)

print(f'Caltech101 total samples: {len(caltech_dataset)}')
print(f'Caltech101 classes: {len(caltech_class_names)}')
display(caltech_split_counts)
display(caltech_train_counts['count'].describe().to_frame('train_class_counts'))
print('Most frequent training classes')
display(caltech_train_counts.sort_values('count', ascending=False).head(10))
print('Least frequent training classes')
display(caltech_train_counts.sort_values('count', ascending=True).head(10))
plot_class_count_histogram(caltech_train_counts['count'], 'Caltech101 training class frequency distribution')
show_sample_grid(caltech_train_raw, class_names=caltech_class_names, title='Caltech101 train samples', n_samples=9)

caltech_base_transform = make_caltech_base_transform(resize=(300, 200))
caltech_train_loader = DataLoader(
    AugmentationWrapper(caltech_train_raw, caltech_base_transform),
    batch_size=128,
    shuffle=False,
    num_workers=0,
)
caltech_mean, caltech_std = mean_and_std_for_normalization(caltech_train_loader)
print('Caltech101 train mean:', np.round(caltech_mean, 6))
print('Caltech101 train std:', np.round(caltech_std, 6))

## Enrico screenshots and wireframes

For UMAP, the notebook uses the raw split first, then the same pixel-level embedding pipeline for screenshots and wireframes so the two modalities stay comparable.

In [ ]:
ENRICO_ROOT = '/kaggle/input/datasets/nazariyyuchnovskiy/enricoscreenshotsandwireframes'

screen_train_raw, screen_val_raw, screen_test_raw = CustomEnricoDataset.create_splits(
    root=ENRICO_ROOT,
    seed=SEED,
    use_wireframes=False,
    transform=None,
    train_transform=None,
    eval_transform=None,
    augment_for_each=None,
    allowed_classes=None,
)
wire_train_raw, wire_val_raw, wire_test_raw = CustomEnricoDataset.create_splits(
    root=ENRICO_ROOT,
    seed=SEED,
    use_wireframes=True,
    transform=None,
    train_transform=None,
    eval_transform=None,
    augment_for_each=None,
    allowed_classes=None,
)

screen_all = ConcatDataset([screen_train_raw, screen_val_raw, screen_test_raw])
wire_all = ConcatDataset([wire_train_raw, wire_val_raw, wire_test_raw])
screen_class_names = [name for name, index in sorted(screen_train_raw.class_to_idx.items(), key=lambda item: item[1])]
wire_class_names = [name for name, index in sorted(wire_train_raw.class_to_idx.items(), key=lambda item: item[1])]

print(f'Screenshot split sizes: {len(screen_train_raw)} / {len(screen_val_raw)} / {len(screen_test_raw)}')
print(f'Wireframe split sizes: {len(wire_train_raw)} / {len(wire_val_raw)} / {len(wire_test_raw)}')
display(split_summary_frame(screen_train_raw, screen_val_raw, screen_test_raw))
display(split_summary_frame(wire_train_raw, wire_val_raw, wire_test_raw))
display(class_count_frame(screen_all, screen_class_names).sort_values('count', ascending=False).head(10))
display(class_count_frame(wire_all, wire_class_names).sort_values('count', ascending=False).head(10))

In [ ]:
screen_features, screen_labels = dataset_to_matrix(screen_all, resize=(48, 48))
wire_features, wire_labels = dataset_to_matrix(wire_all, resize=(48, 48))

screen_coords = project_with_umap(screen_features)
wire_coords = project_with_umap(wire_features)

plot_umap(screen_coords, screen_labels, screen_class_names, 'Enrico screenshots UMAP')
plot_umap(wire_coords, wire_labels, wire_class_names, 'Enrico wireframes UMAP')

screen_summary = summarize_umap(screen_coords, screen_labels, screen_class_names, k=15)
wire_summary = summarize_umap(wire_coords, wire_labels, wire_class_names, k=15)

print_umap_summary(screen_summary, 'Screenshots UMAP summary')
print_umap_summary(wire_summary, 'Wireframes UMAP summary')

## Useful follow-ups for the paper

A few extra visuals or metrics that would fit this notebook without making it heavy: a centroid-distance heatmap, a nearest-neighbor retrieval panel for the most overlapping classes, silhouette or Davies-Bouldin scores per modality, and a compact class-count chart for Caltech101.